In [ ]:
import sys
import warnings
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

_backend_src = Path("../backend/app/src").resolve()
if str(_backend_src) not in sys.path:
    sys.path.insert(0, str(_backend_src))

from herbs_detection.model import predict_top3 as pt_top3
from herbs_detection.model_sklearn import predict_top3 as sk_top3

print("Both models loaded OK")

In [ ]:
# ── Only line to change: point to the folder you want to predict ─────────────
PREDICT_DIR = "../data/dataset_aromatic_plants"

In [ ]:
import asyncio

async def predict_downloaded_images(
    image_dir: str,
    csv_out: str = None,
    max_concurrent: int = 2,
) -> pd.DataFrame:
    """
    Async batch prediction on a folder of images.

    Handles two layouts automatically:
      - Flat folder  → images directly inside image_dir (intended_label = folder name)
      - Sub-folders  → one sub-folder per species

    max_concurrent controls how many images are predicted simultaneously.
    PyTorch releases the GIL during C++ compute, so 2-4 gives a real speedup
    without competing for GPU memory.
    """
    image_dir = Path(image_dir).resolve()

    # ── Collect files (sync, fast) ────────────────────────────────────────────
    flat_images = [
        f for f in image_dir.iterdir()
        if f.is_file() and f.suffix.lower() in (".jpg", ".jpeg", ".png")
    ]
    if flat_images:
        img_files = [(img, image_dir.name) for img in sorted(flat_images)]
    else:
        img_files = []
        for sub in sorted(image_dir.iterdir()):
            if sub.is_dir():
                for img in sorted(sub.iterdir()):
                    if img.suffix.lower() in (".jpg", ".jpeg", ".png"):
                        img_files.append((img, sub.name))

    n_species = len({label for _, label in img_files})
    print(f"Found {len(img_files)} images / {n_species} label(s) in {image_dir}")

    # ── Async prediction ──────────────────────────────────────────────────────
    loop      = asyncio.get_event_loop()
    semaphore = asyncio.Semaphore(max_concurrent)

    def _predict_sync(filepath: Path, intended_label: str):
        """Runs in a thread — both models called here."""
        try:
            pt_preds = pt_top3(str(filepath))
            sk_preds = sk_top3(str(filepath))
        except Exception as e:
            print(f"\n[ERROR] {filepath.name}: {e}")
            return None

        row = {"filepath": str(filepath), "intended_label": intended_label}
        for i, (species, conf) in enumerate(pt_preds, 1):
            row[f"pt_pred{i}"] = species
            row[f"pt_conf{i}"] = conf
        for i, (species, conf) in enumerate(sk_preds, 1):
            row[f"sk_pred{i}"] = species
            row[f"sk_conf{i}"] = conf
        return row

    async def predict_one(filepath: Path, intended_label: str):
        async with semaphore:
            return await loop.run_in_executor(None, _predict_sync, filepath, intended_label)

    tasks = [predict_one(fp, label) for fp, label in img_files]

    records = []
    for future in tqdm(asyncio.as_completed(tasks), total=len(img_files), desc="Predicting"):
        result = await future
        if result:
            records.append(result)

    df = pd.DataFrame(records)
    if csv_out:
        df.to_csv(csv_out, index=False)
        print(f"Saved → {csv_out}")
    return df

In [ ]:
image_dir = Path(PREDICT_DIR).resolve()   # resolve() turns relative → absolute
csv_out   = image_dir / "predictions.csv"

# ── Diagnostic: confirm what the notebook sees ────────────────────────────────
print(f"Scanning : {image_dir}")
print(f"Exists   : {image_dir.exists()}")
entries = list(image_dir.iterdir())
dirs    = [e for e in entries if e.is_dir()]
images  = [e for e in entries if e.suffix.lower() in (".jpg", ".jpeg", ".png")]
print(f"Sub-folders : {len(dirs)}  |  flat images : {len(images)}")
print(f"First 5 sub-folders : {[d.name for d in dirs[:5]]}")
print()

In [ ]:
df = await predict_downloaded_images(str(image_dir), csv_out=str(csv_out))
df.head(10)

In [ ]:
df_allimages = pd.DataFrame()

for row in df.itertuples():
    if row.filepath

In [ ]:
df_allimages = pd.DataFrame()
index_id = []

for idx, row in enumerate(df.itertuples()):
    name = df.loc[idx, "filepath"].split("/")
    if "all_images" in name:
        name = name[-1].split("_")[0]
        df_temp = pd.DataFrame([row])
        df_temp["intended_label"] = name
        df_allimages = pd.concat([df_allimages, df_temp], ignore_index=True)
        index_id.append(idx)


In [ ]:
df_allimages

In [ ]:
df_allimages.to_csv(image_dir / "predictions_allimages.csv", index=False)

In [ ]:
df_without_allimgaes = df.drop(index=index_id)

In [ ]:
df_without_allimgaes.to_csv(image_dir / "predictions_without_allimages.csv", index=False)

In [ ]:
pytorch = 0
sklearn = 0

for row in df_allimages.itertuples():
    if row.intended_label == row.pt_pred1:
        pytorch += 1
    if row.intended_label == row.sk_pred1:
        sklearn += 1

print(f"PyTorch correct predictions: {pytorch/len(df_allimages):.1%} ({pytorch}/{len(df_allimages)})")
print(f"Sklearn correct predictions: {sklearn/len(df_allimages):.1%} ({sklearn}/{len(df_allimages)})")

In [ ]:
pytorch2 = 0
sklearn2 = 0
df_without_allimgaes["pytorch_correct"] = 0
df_without_allimgaes["sklearn_correct"] = 0

for row in df_without_allimgaes.itertuples():
    if "2" in row.intended_label:
        name = row.intended_label.split("2")[0]
    else:
        name = row.intended_label
    if name == row.pt_pred1:
        pytorch2 += 1
        df_without_allimgaes.at[row.Index, "pytorch_correct"] = 1
    if name == row.sk_pred1:
        sklearn2 += 1
        df_without_allimgaes.at[row.Index, "sklearn_correct"] = 1

print(f"PyTorch correct predictions: {pytorch2/len(df_without_allimgaes):.1%} ({pytorch2}/{len(df_without_allimgaes)})")
print(f"Sklearn correct predictions: {sklearn2/len(df_without_allimgaes):.1%} ({sklearn2}/{len(df_without_allimgaes)})")

In [ ]:
count = df_without_allimgaes["intended_label"].value_counts()


In [ ]:
df_test = df_without_allimgaes.groupby("intended_label")["pytorch_correct"].value_counts()

print(df_test.to_string())

In [ ]:
df_test2 = df_without_allimgaes.groupby("intended_label")["sklearn_correct"].value_counts()
print(df_test2.to_string())

In [ ]:
df_test4 = df_without_allimgaes[(df_without_allimgaes["pytorch_correct"] == 1) & (df_without_allimgaes["sklearn_correct"] == 1)]
df_test4.groupby("intended_label").size()

In [ ]:
df_test2 = df_without_allimgaes.groupby("intended_label")[[ "sklearn_correct", "pytorch_correct" ]].value_counts()
print(df_test2.to_string())

In [ ]:
list_images = list(Path("../data/raw/all_images").glob("*.*"))
list_images2 = [img.name for img in list_images]
len(list_images2)

In [ ]:
df_test4.loc[4, "filepath"].split("/")[-1]

In [ ]:
import pandas as pd

labels = pd.read_csv("../data/files_df.csv")
labels.head()

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

target_dir = Path("../data/processed/selected_images_model")
target_dir.mkdir(parents=True, exist_ok=True)

csv_path = Path("../data/files_df.csv")

saved = 0
rows_to_add = []

for row in df_test4.itertuples():
    src = Path(row.filepath)
    if src.name not in list_images2 and src.exists():
        dst = target_dir / src.name
        shutil.copy2(src, dst)
        saved += 1
        print(f"Saved: {src.name}")

        rows_to_add.append({
            "filename": src.name,
            "name": row.pt_pred1
        })

if rows_to_add:
    new_rows = pd.DataFrame(rows_to_add)

    if csv_path.exists() and csv_path.stat().st_size > 0:
        files_df = pd.read_csv(csv_path)
    else:
        files_df = pd.DataFrame(columns=["filename", "name"])

    if "filename" not in files_df.columns:
        files_df["filename"] = pd.NA
    if "name" not in files_df.columns:
        files_df["name"] = pd.NA

    files_df = pd.concat([files_df, new_rows], ignore_index=True)
    files_df = files_df.drop_duplicates(subset=["filename"], keep="last")
    files_df.to_csv(csv_path, index=False)

    print(f"Updated {csv_path.resolve()} with {len(new_rows)} row(s)")
else:
    print("No new images matched the condition.")

print(f"Done. Saved {saved} image(s) to {target_dir.resolve()}")